In [0]:
# Install required package
%pip install googletrans==3.1.0a0


Looking in indexes: [REDACTED]
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.6 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16411 sha256=220691b3316e719b259cb986937931b46923258255fc7581723992ac8de35ebb
  Stored in directory: /home/spark-a45c6e34-bd98-406f-bcaa-2c/.cache/pip/wheels/96/ac/bd/9df9eab356c0576896e97147425987f6f45e9e46456c978d18
Successfully built googletrans
  Attempting uninstall: h11
    Found existing installation: h11 0.16.0
    Not uninstalling h11 at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-a45c6e34-bd98-406f-bcaa-2c3bad6a0e51
    Can't uninstall 'h11'. No files were found to uninstall.
  Attempting uninstall: chardet
    Found existing installation: chardet 4.0.0
    Not uninstalling chardet at /databricks/python3/lib/python3.12/

In [0]:

from googletrans import Translator, LANGUAGES

# Exercise 1: Machine Translation

# Part 1: Basic Translation
translator = Translator()
text_en = "Hello, how are you?"
trans = translator.translate(text_en, src='en', dest='fr')
print(f"{text_en} -> {trans.text}")





Hello, how are you? -> Bonjour comment allez-vous?


In [0]:

# Part 2: Batch Translation Implementation
def batch_translate(texts, src='en', dest='fr'):
    """
    Translate a list of texts from source to destination language.
    
    Args:
        texts (list): List of text strings to translate.
        src (str): Source language code (default: 'en').
        dest (str): Destination language code (default: 'fr').
    
    Returns:
        list: List of dictionaries with original text, translated text, source and destination languages.
    """
    results = []
    for t in texts:
        result = {
            'original_text': t,
            'translated_text': None,
            'source_language': src,
            'destination_language': dest
        }
        try:
            if not t or not isinstance(t, str):
                raise ValueError("Invalid or empty text")
            trans = translator.translate(t, src=src, dest=dest)
            result['translated_text'] = trans.text
        except Exception as e:
            result['translated_text'] = f"Error: {str(e)}"
        results.append(result)
    return results

# Example batch translation
texts = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]
batch_results = batch_translate(texts, src='en', dest='fr')
for res in batch_results:
    print(res)


{'original_text': 'Hello, how are you?', 'translated_text': 'Bonjour comment allez-vous?', 'source_language': 'en', 'destination_language': 'fr'}
{'original_text': 'Bonjour, comment allez-vous?', 'translated_text': 'Bonjour, commentez allez-vous ?', 'source_language': 'en', 'destination_language': 'fr'}
{'original_text': 'Hola, ¿cómo estás?', 'translated_text': 'Hola, comment es-tu ?', 'source_language': 'en', 'destination_language': 'fr'}
{'original_text': 'Guten Tag, wie geht es dir?', 'translated_text': 'Guten Tag, qui est là ?', 'source_language': 'en', 'destination_language': 'fr'}


In [0]:
# Exercise 2: Language Detection

# Part 1: Basic Language Detection
def detect_language(text):
    """
    Detect the language of input text.
    
    Args:
        text (str): Input text.
    
    Returns:
        dict: Detected language code and human readable name.
    """
    try:
        if not text or not isinstance(text, str):
            raise ValueError("Invalid or empty text")
        detected = translator.detect(text)
        lang_code = detected.lang
        lang_name = LANGUAGES.get(lang_code, 'unknown')
        return {
            'detected_language_code': lang_code,
            'detected_language_name': lang_name
        }
    except Exception as e:
        return {'detected_language_code': None, 'detected_language_name': f"Error: {str(e)}"}

for txt in texts:
    detection = detect_language(txt)
    print(f"Detected language: {detection['detected_language_code']} ({detection['detected_language_name']})")

Detected language: en (english)
Detected language: fr (french)
Detected language: es (spanish)
Detected language: de (german)


In [0]:

# Part 2: Integrated Translation System
class LanguageProcessor:
    """
    Provides language detection and translation utilities.
    """
    translator = Translator()
    languages = LANGUAGES

    @staticmethod
    def detect_language(text):
        """
        Detects language code and name for text.

        Args:
            text (str): Input text.

        Returns:
            dict: {'code': str, 'name': str}
        """
        try:
            if not text or not isinstance(text, str):
                raise ValueError("Empty or invalid text")
            detected = LanguageProcessor.translator.detect(text)
            code = detected.lang
            name = LanguageProcessor.languages.get(code, 'unknown')
            return {'code': code, 'name': name}
        except Exception as e:
            return {'code': None, 'name': f"Error: {str(e)}"}

    @staticmethod
    def translate_text(text, dest, src=None):
        """
        Translate single text to desired language.

        Args:
            text (str): Input text.
            dest (str): Destination language code.
            src (str): Source language code. (If None, auto-detect)

        Returns:
            dict: {'original': str, 'translated': str, 'src': str, 'dest': str}
        """
        try:
            if not text or not isinstance(text, str):
                raise ValueError("Empty or invalid text")
            if dest not in LanguageProcessor.languages:
                raise ValueError("Invalid destination language code")
            if src is None:
                detected = LanguageProcessor.translator.detect(text)
                src = detected.lang
            translation = LanguageProcessor.translator.translate(text, src=src, dest=dest)
            return {
                'original': text,
                'translated': translation.text,
                'src': src,
                'dest': dest
            }
        except Exception as e:
            return {
                'original': text,
                'translated': f"Error: {str(e)}",
                'src': src,
                'dest': dest
            }

    @staticmethod
    def process_texts(texts, dest, src=None):
        """
        Process multiple texts with translation and detection.

        Args:
            texts (list): List of text strings.
            dest (str): Destination language code.
            src (str): Source language code. (If None, auto-detect)

        Returns:
            list: List of dicts for each text.
        """
        results = []
        for t in texts:
            lang_det = LanguageProcessor.detect_language(t) if src is None else {'code': src, 'name': LanguageProcessor.languages.get(src, 'unknown')}
            res = LanguageProcessor.translate_text(t, dest, src=lang_det['code'])
            res['detected_language_name'] = lang_det['name']
            results.append(res)
        return results

In [0]:

# Test cases

# 1. Basic translation between different language pairs
print(LanguageProcessor.translate_text("Bonjour, comment allez-vous?", dest="en", src="fr"))

# 2. Multiple texts in different source languages (auto-detect)
multi_trans = LanguageProcessor.process_texts(texts, dest="en")
for r in multi_trans:
    print(r)

# 3. Error cases
invalid_code = LanguageProcessor.translate_text("hello", dest="xx")
empty_text = LanguageProcessor.translate_text("", dest="fr")
long_text = LanguageProcessor.translate_text("A" * 1200, dest="fr")
print(invalid_code)
print(empty_text)
print(long_text)

{'original': 'Bonjour, comment allez-vous?', 'translated': 'Good morning, how are you doing?', 'src': 'fr', 'dest': 'en'}
{'original': 'Hello, how are you?', 'translated': 'Hello, how are you?', 'src': 'en', 'dest': 'en', 'detected_language_name': 'english'}
{'original': 'Bonjour, comment allez-vous?', 'translated': 'Good morning, how are you doing?', 'src': 'fr', 'dest': 'en', 'detected_language_name': 'french'}
{'original': 'Hola, ¿cómo estás?', 'translated': 'Hello how are you?', 'src': 'es', 'dest': 'en', 'detected_language_name': 'spanish'}
{'original': 'Guten Tag, wie geht es dir?', 'translated': 'Good day how are you?', 'src': 'de', 'dest': 'en', 'detected_language_name': 'german'}
{'original': 'hello', 'translated': 'Error: Invalid destination language code', 'src': None, 'dest': 'xx'}
{'original': '', 'translated': 'Error: Empty or invalid text', 'src': None, 'dest': 'fr'}
{'original': 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA